In [ ]:
import os
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Load environment variables from .env file
load_dotenv()

BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
ADAPTER_PATH = r"/kaggle/input/adapter-lora/llama32_1b_vi_sum_lora_adapter"

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Please configure it in .env file")

PROMPT_PREFIX = "Hãy tóm tắt ngắn gọn và súc tích đoạn văn sau:\n"
PROMPT_SUFFIX = "\n\nTóm tắt: "

MAX_NEW_TOKENS = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

2025-11-22 17:34:02.808761: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763832842.972348      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763832843.021537      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Device: cuda


In [3]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 55.3 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.0 MB/s eta 0:00:0000:0100:01
  Attempting unin

In [5]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import torch


model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    token=HF_TOKEN, 
    trust_remote_code=True,
)
print('Base model loaded')

print(f"Loading tokenizer from base model: {BASE_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID, 
    token=HF_TOKEN,
    use_fast=False
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print('Tokenizer loaded')

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Base model loaded
Loading tokenizer from base model: meta-llama/Llama-3.2-1B


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tokenizer loaded


In [6]:
print('Attaching LoRA adapter from', ADAPTER_PATH)
model = PeftModel.from_pretrained(model, ADAPTER_PATH, torch_dtype=torch.bfloat16)
model.eval()
model = model.to(device)
print('Adapter attached and model set to eval mode on', device)

Attaching LoRA adapter from /kaggle/input/adapter-lora/llama32_1b_vi_sum_lora_adapter


/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Adapter attached and model set to eval mode on cuda


In [7]:
!pip install sacrebleu rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.8 MB/s eta 0:00:00


In [11]:
import pandas as pd
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm
import torch
import sacrebleu  

benchmark_path = '/kaggle/input/benchmark-tm-tt/Benchmark/abstractive_summarization_benchmark.csv'
df_benchmark = pd.read_csv(benchmark_path)
df_benchmark = df_benchmark.head(1000)

print(f"Benchmark dataset size: {len(df_benchmark)}")
print(f"Columns: {df_benchmark.columns.tolist()}")

# Cấu hình Tokenizer để chạy Batch (Quan trọng cho Decoder-only models)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Khi generate, padding nên ở bên trái để không làm rối model
tokenizer.padding_side = "left" 

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

BATCH_SIZE = 8 
results = []



num_batches = (len(df_benchmark) + BATCH_SIZE - 1) // BATCH_SIZE

# --- 2. VÒNG LẶP ĐÁNH GIÁ ---
for batch_idx in tqdm(range(num_batches), desc="Evaluating Batches"):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min((batch_idx + 1) * BATCH_SIZE, len(df_benchmark))
    batch_data = df_benchmark.iloc[start_idx:end_idx]
    
    batch_contents = batch_data['content'].tolist()
    batch_references = batch_data['summary'].tolist()
    
    # Tạo Prompt
    batch_prompts = [
        PROMPT_PREFIX + str(content).strip() + PROMPT_SUFFIX 
        for content in batch_contents
    ]
    
    inputs = tokenizer(
        batch_prompts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True,
        max_length=750
    ).to(device)
    
    input_token_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=5,
            early_stopping=True,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    if outputs.shape[1] > input_token_len:
        generated_tokens = outputs[:, input_token_len:]
    else:
        generated_tokens = outputs

    generated_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    
    # --- TÍNH ĐIỂM ---
    for i in range(len(generated_texts)):
        predicted_summary = generated_texts[i].strip()
        reference_summary = str(batch_references[i]) if batch_references[i] is not None else ""
        actual_idx = start_idx + i
        
        # 1. Tính ROUGE
        rouge_scores = scorer.score(str(reference_summary), str(predicted_summary))
        
        # 2. Tính BLEU (Thêm mới)
        # sacrebleu yêu cầu reference dạng list of list (cho multi-ref), ở đây ta có 1 ref
        bleu_score = sacrebleu.sentence_bleu(predicted_summary, [reference_summary]).score

        results.append({
            'idx': actual_idx,
            'reference': reference_summary,
            'predicted': predicted_summary,
            'bleu': bleu_score,
            'rouge1_precision': rouge_scores['rouge1'].precision,
            'rouge1_recall': rouge_scores['rouge1'].recall,
            'rouge1_fmeasure': rouge_scores['rouge1'].fmeasure,
            'rougeL_precision': rouge_scores['rougeL'].precision,
            'rougeL_recall': rouge_scores['rougeL'].recall,
            'rougeL_fmeasure': rouge_scores['rougeL'].fmeasure,
        })

results_df = pd.DataFrame(results)

print("\n" + "="*80)
print(f"OVERALL METRICS SUMMARY")
print("="*80)

print(f"\nBLEU SCORE:")
print(f"  Avg Score:     {results_df['bleu'].mean():.4f}")
print(f"  Std Score:     {results_df['bleu'].std():.4f}")

print(f"\nROUGE-1:")
print(f"  Avg Precision: {results_df['rouge1_precision'].mean():.4f}")
print(f"  Avg Recall:    {results_df['rouge1_recall'].mean():.4f}")
print(f"  Avg F1-Score:  {results_df['rouge1_fmeasure'].mean():.4f}")

print(f"\nROUGE-L:")
print(f"  Avg Precision: {results_df['rougeL_precision'].mean():.4f}")
print(f"  Avg Recall:    {results_df['rougeL_recall'].mean():.4f}")
print(f"  Avg F1-Score:  {results_df['rougeL_fmeasure'].mean():.4f}")

# Lưu file
results_csv_path = 'benchmark_results_beam_search_k2_with_bleu.csv'
results_df.to_csv(results_csv_path, index=False)
print(f"\nResults saved to: {results_csv_path}")
print(f"Total samples evaluated: {len(results_df)}")

Benchmark dataset size: 1000
Columns: ['content', 'summary']


Evaluating Batches:   2%|▏         | 2/125 [02:07<2:10:34, 63.70s/it]


KeyboardInterrupt: 